### Instruction Finetuning

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import socket

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
# from importlib.metadata import version
# pkgs = [
#     "numpy",
#     "matplotlib",
#     "tiktoken",
#     "torch",
#     "tqdm",
#     # "tensorflow"
# ]
# for p in pkgs:
#     print(f"{p} version: {version(p)}")

In [6]:
import torch
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [8]:
if IS_PACE_SERVER:
    from model_training.post_training.artifacts.data.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
    from model_training.post_training.artifacts.data.instruction_tuning.prompt_style.styles import AlpacaFormat
    from model_training.post_training.artifacts.data.instruction_tuning.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader
else:
    ## Fix this import
    raise Exception("Not Running on Pace")
        # from src.model import GPT

### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [9]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(ROOT_DATASET_DIR / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [10]:
alpaca = AlpacaFormat(data)

In [11]:
## Example of format
print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
print("=============================================")
print(alpaca.detailed_response(out_text = data[50]['output']))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

 #### Input:
Ocassion


### Response:
The correct spelling is 'Occasion.'


In [12]:
sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
print(sample)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


#### Spliting the data 

In [13]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
pad_token_id

50256

In [15]:
dataloader = InstructionDataset(ds=data, tokenizer=tokenizer, prompt_style=alpaca)
(sample)

"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is an antonym of 'complicated'?\n\n### Response:\nAn antonym of 'complicated' is 'simple'."

#### Sample Data Prepare

In [16]:

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
custom_collate_fn(batch, pad_token_id=pad_token_id)

(tensor([[    0,     1,     2,     3,     4],
         [    5,     6, 50256, 50256, 50256],
         [    7,     8,     9, 50256, 50256]]),
 tensor([[    1,     2,     3,     4, 50256],
         [    6, 50256,  -100,  -100,  -100],
         [    8,     9, 50256,  -100,  -100]]))

In [17]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


### Creating, Training and Testing Dataset

In [18]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [19]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer, prompt_style=alpaca)

train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_dataset = InstructionDataset(val_data, tokenizer, prompt_style=alpaca)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer, prompt_style=alpaca)
test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [20]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 60]) torch.Size([8, 60])


### Step-4: - Loading the Model Testing inference

In [ ]:
from model_training.post_training.artifacts.model.hf_model.gpt2 import return_model
from model_training.artifacts.model.load_model.gpt2.configs import load_model_config
from model_training.artifacts.model.components.inference.inference import gen_sample
from model_training.artifacts.model.tokenizer.hf_tokenizer import text_to_token_ids, token_ids_to_text

In [133]:
config =load_model_config(dataset_max_len=1024)
MODEL_NAME = 'gpt2'
model = return_model(MODEL_NAME)

# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [134]:
# import inspect
# inspect.signature(tokenizer.encode)

In [135]:
model.eval()
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, allowed_special = {"<|endoftext|>"})

In [136]:
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)

In [150]:
print(token_ids_to_text(out_tokens, tokenizer))

Every effort moves you, and it's not one you can control. And it will be, the moment you are ready to move, you will move. So that you go as fast as you can


### SFT-Training

In [138]:
sample

"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is an antonym of 'complicated'?\n\n### Response:\nAn antonym of 'complicated' is 'simple'."

In [139]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

4.529170989990234


In [140]:
from model_training.artifacts.train.core.train import train_model
from model_training.artifacts.train.core.config import TrainConfig
from model_training.artifacts.train.utils.optimizer import configure_optimizer
from model_training.artifacts.train.utils.lr_scheduler import cosine_lr

In [141]:
torch.manual_seed(123)
optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

num decayed parameter tensors: 50, with 124,318,464 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: True


In [142]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )

In [143]:
from dataclasses import  dataclass
from typing import Any

@dataclass
class TrainComponents:
    optimizer:Any
    dataloader:Any
    scheduler:Any
    
train_comp = TrainComponents(optimizer=optimizer , dataloader=train_loader, scheduler= cosine_lr)
train_comp

TrainComponents(optimizer=AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 0.0003
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 0.0003
    maximize: False
    weight_decay: 0.0
), dataloader=<model_training.post_training.artifacts.data.instruction_tuning.dataloader.CyclingDataLoader object at 0x15536871e480>, scheduler=<function cosine_lr at 0x15537b5d9080>)

In [ ]:
train_config = TrainConfig(num_steps=100, 
                        learning_rate=3e-4, 
                        total_batch_size= total_batch_size,
                        micro_batch_size =micro_batch_size, 
                        seq_length = seq_length, 
                        max_lr = 6e-4, 
                        min_lr =  6e-4 * 0.1,
                        warmup_steps= 715, 
                        accumulation_steps = grad_accum_steps,
                        # max_steps = 19073   # 19,073 steps is ~1 epoch, if data is 10B tokens and batch size 0.5M tokens
                        device=device)

In [156]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [157]:
print(train_config.accumulation_steps)

2


In [ ]:
model = train_model(model, train_components=train_comp, train_config=train_config)

Current Device: cuda
Model set to Training mode:True


AttributeError: 'function' object has no attribute 'parameters'

In [ ]:
text_1 = "What is your name ? "
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, allowed_special = {"<|endoftext|>"})
in_tokens=in_tokens.to(device)
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

What is your name ?  
### Last Name:
What is your occupation?
When you're young, is it in your job you do or will you continue?
When you're
